# Lesson 3 ? Correct Function-Tool Round Trip

Use two strict Harborlight function tools:

- list_upcoming_renewals(days: int)
- calculate_premium_change(current_cents: int, renewal_cents: int)

**The common bug:** running a Python function locally is not enough. The tool result must be returned to the model as function_call_output with the matching call_id before requesting the final answer.

In [ ]:
import os
from types import SimpleNamespace

from openai import OpenAI

from harborlight_responses.config import api_key_available, resolve_model
from harborlight_responses.fixtures import FIXTURE_LABEL
from harborlight_responses.tool_loop import run_tool_loop

selection = resolve_model()
live_requested = os.getenv("HARBORLIGHT_LIVE") == "1"
live = live_requested and api_key_available()
prompt = (
    "Which fictional Harborlight policies renew in the next 30 days? "
    "Use the tools, and calculate the proposed change for Fictional Beacon Books."
)
instructions = (
    "Use the Harborlight tools for every renewal fact and premium calculation. "
    "Return a concise account-manager briefing."
)

if live:
    client = OpenAI()
else:
    first = SimpleNamespace(
        id="temporary-demo-first",
        output=[
            SimpleNamespace(
                type="function_call",
                name="list_upcoming_renewals",
                call_id="demo-call-renewals",
                arguments='{"days": 30}',
            ),
            SimpleNamespace(
                type="function_call",
                name="calculate_premium_change",
                call_id="demo-call-premium",
                arguments='{"current_cents": 126000, "renewal_cents": 132300}',
            ),
        ],
    )
    final = SimpleNamespace(
        id="temporary-demo-final",
        output=[],
        output_text=(
            "Three fictional policies renew in the 30-day window. "
            "Fictional Beacon Books has a verified 5.00% proposed increase."
        ),
    )

    class DemoResponses:
        def __init__(self):
            self.calls = []

        def create(self, **kwargs):
            self.calls.append(kwargs)
            return first if len(self.calls) == 1 else final

    demo_responses = DemoResponses()
    client = SimpleNamespace(responses=demo_responses)
    print(FIXTURE_LABEL)

result = run_tool_loop(
    client,
    model=selection.model,
    input=prompt,
    instructions=instructions,
    mode="live" if live else "fixture",
)

print("Selected model:", selection.model)
for number, execution in enumerate(result.executions, start=1):
    print()
    print(f"Tool {number}: {execution.name}")
    print("Validated arguments:", execution.arguments)
    print("Deterministic output:", execution.output)
print()
print("Final generated answer:")
print(result.response.output_text)

In [ ]:
if not live:
    continuation = demo_responses.calls[1]
    returned_items = continuation["input"]
    assert all(item["type"] == "function_call_output" for item in returned_items)
    assert {item["call_id"] for item in returned_items} == {
        "demo-call-renewals",
        "demo-call-premium",
    }
    print("Verified: both local results were returned as function_call_output.")

## Error behavior and stopping

Arguments are JSON-decoded, validated with strict Pydantic models, and then passed to deterministic services. Unknown tools, invalid arguments, and tool exceptions become structured tool outputs so the model can respond. max_rounds prevents an accidental infinite loop. The same loop handles zero, one, or multiple calls per response.